# Experiment Comparison (ANN vs CNN)

This notebook compares experiment histories and evaluation metrics across models.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data.data_loader import build_tf_datasets
from src.evaluation.metrics import evaluate_models, save_evaluation
from src.models import build_ann_model, build_cnn_model
from src.training.trainer import train_model
from src.utils.config import DEFAULT_DATA_CONFIG


In [ ]:
datasets = build_tf_datasets(DEFAULT_DATA_CONFIG)
models = {"ann": build_ann_model(DEFAULT_DATA_CONFIG), "cnn": build_cnn_model(DEFAULT_DATA_CONFIG)}
histories = {}
for name, model in models.items():
    history, _ = train_model(model, datasets, experiment_name=name)
    histories[name] = history


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history.history["val_accuracy"], label=name)
    axes[1].plot(history.history["val_loss"], label=name)
axes[0].set_title("Validation Accuracy")
axes[1].set_title("Validation Loss")
for ax in axes:
    ax.legend()
plt.show()


In [ ]:
results = evaluate_models(models, datasets["test"])
save_evaluation(results, Path("reports") / "phase4_results.json")
pd.DataFrame({name: {"loss": metrics["loss"], "accuracy": metrics["accuracy"]}
             for name, metrics in results.items()}).T


In [ ]:
for name, metrics in results.items():
    print(f"{name} confusion matrix:\n", metrics["confusion_matrix"])
    print(f"{name} classification report:\n", metrics["classification_report"])
